In [46]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns
import gensim as gs
from gensim.utils import simple_preprocess
from gensim.corpora import Dictionary
from gensim.models import LogEntropyModel, HdpModel, LdaSeqModel, LdaModel, Nmf, TfidfModel
from gensim.models.normmodel import NormModel

class GensimConverter: 

    def __init__(self, texts):
        self.texts = texts
        self.corpus = [simple_preprocess(text) for text in self.texts]
        
        # Switches
        self.has_dct = 0
        self.has_bow = 0
        self.has_model = 0
    
    def text_to_dict(self):
        self.dct = Dictionary(self.corpus)
        self.has_dct = 1

    def corpus_to_bow(self):
        if not self.dct:
            self.text_to_dict()
        self.bow = [self.dct.doc2bow(doc) for doc in self.corpus]
        self.has_bow = 1

    def bow_to_model(self, m='LogEntropyModel'):
        if not self.bow:
            self.corpus_to_bow()
        f = getattr(gs.models, m)
        self.model = f(self.bow, normalize=True)
        self.modeled_corpus = self.model[self.bow]
        self.has_model = 1
        
    def gs_to_eta(self):
        self.TOKEN = pd.DataFrame(self.corpus).stack().to_frame('term_str')
        self.TOKEN.index.names = ['doc_id','token_num']
        if self.has_dct:
            self.VOCAB = pd.DataFrame([item for item in self.dct.items()])
            self.VOCAB.columns = ['term_id', 'term_str']
            self.VOCAB = self.VOCAB.set_index('term_id')
        if self.has_model:
            self.WEIGHTED = pd.concat([
                pd.DataFrame(doc)\
                    .reset_index(drop=True)\
                    .set_index(0) 
                for doc in self.modeled_corpus], 
                keys=[i for i in range(len(self.modeled_corpus))])\
                .unstack(fill_value=0)
            self.WEIGHTED.columns = pv.dct.values()
            self.WEIGHTED.index.name = 'doc_id'

In [47]:
import ipywidgets as widgets
from ipywidgets import interact

In [ ]:
# src_id = "christenson_ximenez"
@interact(
    norm=['l1','l2'],
    model=['lda','nmf','nmf+norm','nmf+logent'],
    kappa=(.01,1.,.01),
    n_topics=(1,10,1),
    no_below=(0, 10, 1),
    no_above=(.25, 1., .05),
    src_id = ['colop','ximenez','christenson','christenson_ximenez'])
def show_narrative(
    src_id = 'christenson',
    n_topics = 6,
    no_below = 0,
    no_above = .4,
    norm='l2',
    model='nmf+norm',
    kappa=.1):

    CHUNK = pd.read_csv(f"../{src_id}/{src_id}-CHUNK-60.csv").set_index('chunk_num').sort_index()

    pv = GensimConverter(CHUNK.chunk_str.to_list())
    pv.text_to_dict()
    pv.dct.filter_extremes(no_below=no_below, no_above=no_above)
    print("V:", len(pv.dct.items()))
    pv.corpus_to_bow()

    tfidf_model = TfidfModel(pv.bow)
    tfidf_corpus = tfidf_model[pv.bow]

    norm_model = NormModel(tfidf_corpus, norm='l2')
    norm_corpus = [norm_model[doc] for doc in tfidf_corpus]

    logent_model = LogEntropyModel(pv.bow)
    logent_corpus = logent_model[pv.bow]

    if model == 'lda':
        lda = LdaModel(pv.bow, n_topics, alpha='auto', eta='auto')
    elif model == 'nmf':
        lda = Nmf(tfidf_corpus, n_topics, kappa=kappa)
    elif model == 'nmf+norm':
        lda = Nmf(norm_corpus, n_topics, kappa=kappa)
    elif model == 'nmf+logent':
        lda = Nmf(logent_corpus, n_topics, kappa=kappa)
    else:
        return "No model"

    PHI = pd.DataFrame(lda.get_topics(), columns=pv.dct.values())

    dtm = []
    keys = [i for i in range(len(pv.bow))]
    for i in keys:
        dtm.append(pd.DataFrame(lda.get_document_topics(pv.bow[i])))
    DTM = pd.concat(dtm, keys=keys)
    DTM = DTM.reset_index()
    DTM = DTM.set_index(['level_0', 0])[1]
    DTM.index.names = ['doc_id', 'topic_id']
    DTM = DTM.unstack(fill_value=0)
    # DTM

    PHI = pd.DataFrame(lda.get_topics())
    PHI.columns = pv.dct.values()
    # PHI

    TOPIC_INFO = pd.DataFrame(lda.show_topics())[1].str.split('+', expand=True).stack().str.split('*', expand=True)
    TOPIC_INFO.index.names = ['topic_id', 'term_rank']
    TOPIC_INFO.columns = ['topic_weight', 'term_id']
    TOPIC_INFO.term_id = TOPIC_INFO.term_id.str.replace('"','').str.strip().astype(int)
    TOPIC_INFO['term_str'] = TOPIC_INFO.term_id.apply(lambda x: pv.dct.id2token[x])

    TOPIC = TOPIC_INFO.groupby('topic_id').term_str.apply(lambda x: " ".join(x)).to_frame('top_terms')
    # print(TOPIC)

    fig, ax = plt.subplots(figsize=(15, n_topics * .3))
    sns.heatmap(DTM.T.set_index(TOPIC.top_terms), cmap="Spectral", center=0)
    plt.show()

interactive(children=(Dropdown(description='src_id', index=2, options=('colop', 'ximenez', 'christenson', 'chr…